# Project 3: Sports Analytics
## Player Performance & Match Prediction

In [4]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')
from utils.data_analysis_utils import utils
print("✅ Imports OK")


✅ Imports OK


In [ ]:
np.random.seed(42)
teams   = ['Lakers','Warriors','Celtics','Bucks','Nuggets','Suns','Heat','76ers']
n_games = 2500
pos_base = {
    'PG':{'pts':15,'ast':7,'reb':4,'stl':1.5,'blk':0.3},
    'SG':{'pts':18,'ast':4,'reb':5,'stl':1.2,'blk':0.4},
    'SF':{'pts':16,'ast':3.5,'reb':6,'stl':1.0,'blk':0.6},
    'PF':{'pts':14,'ast':2.5,'reb':8,'stl':0.8,'blk':1.0},
    'C': {'pts':12,'ast':2,'reb':10,'stl':0.6,'blk':1.5},
}

players = []
for t in teams:
    for j in range(15):
        pos = np.random.choice(['PG','SG','SF','PF','C'])
        exp = int(np.clip(np.random.normal(5,3), 0, 18))
        players.append({'PlayerID':f'{t}_{j}','Name':f'Player_{t}_{j}',
                        'Team':t,'Position':pos,
                        'Age': int(np.clip(np.random.normal(26,4), 19, 38)),
                        'Experience_Years':exp})
players_df = pd.DataFrame(players)

rows = []
for g in range(n_games):
    home, away = np.random.choice(teams, 2, replace=False)
    home_pl = players_df[players_df['Team']==home].sample(10)
    away_pl = players_df[players_df['Team']==away].sample(10)
    for _, p in pd.concat([home_pl, away_pl]).iterrows():
        b  = pos_base[p['Position']]
        ef = 1 + p['Experience_Years']/50
        rows.append({
            'GameID':g+1,'PlayerID':p['PlayerID'],'Team':p['Team'],
            'Home_Team':home,'Away_Team':away,
            'Is_Home':int(p['Team']==home),
            'Points':  round(max(0,np.random.normal(b['pts']*ef,5)),1),
            'Assists': round(max(0,np.random.normal(b['ast']*ef,2)),1),
            'Rebounds':round(max(0,np.random.normal(b['reb']*ef,2)),1),
            'Steals':  round(max(0,np.random.normal(b['stl'],0.8)),1),
            'Blocks':  round(max(0,np.random.normal(b['blk'],0.7)),1),
            'Turnovers':round(max(0,np.random.gamma(1.5,0.8)),1),
            'Minutes': round(np.random.uniform(15,38),1),
            'FieldGoal_Pct':round(np.random.beta(8,6)*100,1),
            'ThreePoint_Pct':round(np.random.beta(6,8)*100,1),
            'Position':p['Position'],'Age':p['Age'],
            'Experience_Years':p['Experience_Years'],
        })

gdf = pd.DataFrame(rows)
totals = gdf.groupby(['GameID','Team'])['Points'].sum()
winners= totals.groupby('GameID').idxmax().apply(lambda x: x[1])
gdf['Winner']   = gdf['GameID'].map(winners)
gdf['Team_Won'] = (gdf['Team'] == gdf['Winner']).astype(int)
print(f"✅ {len(gdf):,} player-game records  |  {n_games} games")


AttributeError: 'float' object has no attribute 'clip'

In [ ]:
os.makedirs('visualizations', exist_ok=True)
pos_order = ['PG','SG','SF','PF','C']

fig, axes = plt.subplots(2, 2, figsize=(14,10))
for ax, col, title in zip(
        [axes[0,0],axes[0,1],axes[1,0]],
        ['Points','Assists','Rebounds'],
        ['Points by Position','Assists by Position','Rebounds by Position']):
    ax.boxplot([gdf[gdf['Position']==p][col] for p in pos_order], labels=pos_order)
    ax.set_title(title, fontweight='bold'); ax.grid(True,alpha=0.3)

win_rate = gdf.groupby('Position')['Team_Won'].mean()*100
axes[1,1].bar(pos_order, [win_rate.get(p,0) for p in pos_order], color='teal')
axes[1,1].axhline(50, color='red', linestyle='--', label='50%')
axes[1,1].set_title('Win Rate by Position', fontweight='bold'); axes[1,1].legend()
plt.tight_layout()
plt.savefig('visualizations/position_performance.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: position_performance.png")


In [ ]:
# Win prediction model
feats = ['Points','Assists','Rebounds','Steals','Blocks',
         'FieldGoal_Pct','ThreePoint_Pct','Minutes','Experience_Years']
X = gdf[feats].fillna(gdf[feats].mean())
y = gdf['Team_Won']
sc = StandardScaler()
X_sc = sc.fit_transform(X)
X_tr,X_te,y_tr,y_te = train_test_split(X_sc,y,test_size=0.2,random_state=42)

rf = RandomForestClassifier(n_estimators=100,random_state=42,n_jobs=-1)
rf.fit(X_tr,y_tr)
acc = accuracy_score(y_te, rf.predict(X_te))
print(f"Win Prediction Accuracy: {acc*100:.1f}%")

fi = pd.Series(rf.feature_importances_, index=feats).sort_values()
fig, ax = plt.subplots(figsize=(10,6))
fi.plot.barh(ax=ax, color='steelblue')
ax.set_title('Feature Importance for Win Prediction', fontweight='bold')
plt.tight_layout()
plt.savefig('visualizations/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: feature_importance.png")


In [ ]:
# Team performance
team_stats = gdf.groupby('Team').agg(
    Win_Rate=('Team_Won','mean'),
    Avg_Pts =('Points','mean'),
    Avg_Ast =('Assists','mean')
).round(2)
team_stats['Win_Rate'] *= 100

fig, axes = plt.subplots(1, 2, figsize=(14,6))
team_stats.sort_values('Win_Rate').plot.barh(y='Win_Rate', ax=axes[0], color='gold', legend=False)
axes[0].set_title('Team Win Rates', fontweight='bold'); axes[0].axvline(50,color='red',linestyle='--')
axes[1].scatter(team_stats['Avg_Pts'], team_stats['Win_Rate'], s=200)
for t,r in team_stats.iterrows():
    axes[1].annotate(t,(r['Avg_Pts'],r['Win_Rate']))
axes[1].set_xlabel('Avg Points'); axes[1].set_ylabel('Win Rate (%)')
axes[1].set_title('Points vs Win Rate', fontweight='bold'); axes[1].grid(True,alpha=0.3)
plt.tight_layout()
plt.savefig('visualizations/team_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

utils.plot_correlation_matrix(gdf[feats], save_path='visualizations/correlation_matrix.png')
gdf.to_csv('processed_sports_data.csv', index=False)
print("✅ Project 3 complete.")
